# 🎛️ Run a Strategy

A reusable runner: **see the strategies available → pick one → run a backtest → (optionally) log it to RDS.**

How to use it: edit the one **CONFIG** cell, then *Run All*. Everything else is automatic.
Results you log here show up in **`explore_results.ipynb`**.

In [ ]:
# --- setup: make `ehf` importable, load .env (FMP + AWS creds), connect the registry ---
%matplotlib inline
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent          # walk up to the repo root so `import ehf...` works
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from dotenv import load_dotenv
load_dotenv(_root / ".env")        # FMP_API_KEY + AWS creds + EHF_* backend config

import pandas as pd
from ehf.research import strategies                  # importing registers all strategies
from ehf.research.strategies.base import _REGISTRY
from ehf.research.config.spec import RunSpec, UniverseSpec, CostConfig, FillModel
from ehf.research.data.fmp_provider import FMPProvider
from ehf.research.backtest.engine import run_backtest
from ehf.research.results import ExperimentRegistry
from ehf.research.data import Timeframe

registry = ExperimentRegistry()    # -> RDS+S3 when .env configures it, else local
print("results backend :", type(registry.store).__name__)
print("strategies      :", ", ".join(sorted(_REGISTRY)))

## 1 · The strategy menu

| key | what it trades | family |
|---|---|---|
| `tsmom` | time-series momentum, vol-scaled (Moskowitz-Ooi-Pedersen) | trend |
| `tsmom_regime` | `tsmom`, gated to *trending* regimes (permutation-entropy + Hurst) | trend + complexity |
| `xsmom` | cross-sectional: long winners / short losers, dollar-neutral | trend |
| `rise_fall` | asymmetric ATR-anchored dip-buy inside an uptrend | mean reversion |
| `bollinger_reversion` | buy extreme deviations below a long Bollinger band | mean reversion |

The table below is generated live — every column after `class` is a **tunable param** you can override in CONFIG.

In [ ]:
# Live menu: each strategy and its default (overridable) params.
pd.DataFrame(
    [{"strategy": k, "class": c.__name__, **c.defaults} for k, c in sorted(_REGISTRY.items())]
).set_index("strategy")

## 2 · Configure your run  ✏️

**This is the only cell you edit.** Pick a strategy, a universe, dates, and any param overrides.

In [ ]:
# ============================================================================
#   EDIT THIS CELL, then "Run All". Everything below is automatic.
# ============================================================================
STRATEGY = "bollinger_reversion"          # one of the keys in the menu above
START    = "2024-01-04"
END      = "2025-12-31"
SYMBOLS = FMPProvider().random_symbols(250, as_of=START)

PARAMS   = {
    "window": 30,
    "n_std": 1.5,
    "target_pct": 0.05,
    "stop_pct": 0.1,
    "max_hold": 90
}                      # override the chosen strategy's params, e.g.
                                   #   {"lookback": 126, "hurst_min": 0.45}

SLIPPAGE_BPS = 1.5                # per-trade cost assumption (basis points)
FILL         = "next_open"         # next_open (safe) | next_close | same_close (debug)
LOG_TO_RDS   = True                # persist so it appears in explore_results.ipynb

## 3 · Fetch data & run

Builds the reproducible `RunSpec` (its hash is the permanent ID), pulls cached OHLCV, and runs the look-ahead-safe, cost-aware engine.

In [ ]:
timeframe = Timeframe.HOUR
spec = RunSpec(
    strategy=STRATEGY,
    params=PARAMS,
    universe=UniverseSpec(symbols=tuple(s.upper() for s in SYMBOLS)),
    start=START, end=END,
    costs=CostConfig(slippage_bps=SLIPPAGE_BPS),
    fill_model=FillModel(FILL),
    timeframe=timeframe
)
bars = FMPProvider().get_bars(list(SYMBOLS), START, END, timeframe=timeframe)   # cached after first fetch
result = run_backtest(spec, bars)

# Pretty metrics table (percentages where it makes sense).
_pct = {"cagr", "total_return", "ann_vol", "max_drawdown", "hit_rate"}
_order = ["cagr", "sharpe", "sortino", "calmar", "max_drawdown", "ann_vol",
          "total_return", "hit_rate", "n_trades", "avg_turnover", "n_bars"]
rows = [(k, f"{result.metrics[k]:.2%}" if k in _pct else f"{result.metrics[k]:,.2f}")
        for k in _order if k in result.metrics]
print(f"{STRATEGY}  ·  {','.join(SYMBOLS)}  ·  {START}..{END}")
print("config_hash:", spec.config_hash)
pd.DataFrame(rows, columns=["metric", "value"]).set_index("metric")

## 4 · Visualize

Equity curve + drawdown for this run (no need to log it first).

In [ ]:
import matplotlib.pyplot as plt
eq = result.equity.set_index("timestamp")["equity"]
dd = eq / eq.cummax() - 1.0
fig, ax = plt.subplots(2, 1, figsize=(12, 7), height_ratios=[3, 2], sharex=True)
ax[0].plot(eq.index, eq, lw=1.3); ax[0].set_title(f"Equity — {STRATEGY}"); ax[0].grid(alpha=.3)
ax[0].yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax[1].fill_between(dd.index, dd, 0, color="crimson", alpha=.4)
ax[1].set_title("Drawdown"); ax[1].grid(alpha=.3)
ax[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1.0))
fig.tight_layout(); plt.show()

## 5 · Log to RDS

Persist the run (metadata → RDS, equity/trades → S3). Idempotent: same config = same hash, never duplicated.

In [ ]:
if LOG_TO_RDS:
    h = registry.log_run(spec, result)
    print(f"✅ logged to {type(registry.store).__name__}: {h}")
    print("   find it in explore_results.ipynb, or:  registry.get_run(%r)" % h)
else:
    print("not logged (LOG_TO_RDS = False)")